# Module 2 — Spin Efficiency

Magnus-force spin efficiency per pitcher × pitch type.

**Formula (Nathan 2008):**



High efficiency = spin is oriented to maximize Magnus-force movement.  
Low efficiency = gyro spin component is present (slider, cutter).

In [ ]:
import sys
sys.path.insert(0, "../modules")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ingest import pull_statcast_range
from spin_efficiency import (
    compute_spin_efficiency,
    aggregate_pitcher_pitch_type,
    apply_reliability_weighting,
    normalize_spin_efficiency,
    build_pitcher_spin_profile,
    MAGNUS_K,
)

%matplotlib inline
pd.set_option("display.max_columns", 50)

In [ ]:
# Load cached test data (2023-06-01 to 2023-06-07)
df = pull_statcast_range("2023-06-01", "2023-06-07", label="test")
print(f"{len(df):,} pitches loaded")

In [ ]:
# Compute pitch-level spin efficiency
df_se = compute_spin_efficiency(df)
df_se[["pitch_type", "release_speed", "release_spin_rate",
        "movement_mag", "theoretical_max_mov", "spin_efficiency"]].dropna().head(10)

In [ ]:
# Spin efficiency by pitch type
by_type = (
    df_se[df_se["spin_efficiency"].notna()]
    .groupby("pitch_type")["spin_efficiency"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
    .round(3)
)
by_type

In [ ]:
# Visualise SE distribution by pitch type
import matplotlib.pyplot as plt

pitch_order = ["CU", "KC", "FF", "SI", "CH", "FC", "FS", "SL", "ST", "SV"]
plot_types = [p for p in pitch_order if p in df_se["pitch_type"].values]

fig, ax = plt.subplots(figsize=(12, 5))
data = [df_se[df_se["pitch_type"] == pt]["spin_efficiency"].dropna().values for pt in plot_types]
ax.boxplot(data, labels=plot_types, patch_artist=True)
ax.set_ylabel("Spin Efficiency")
ax.set_title("Spin Efficiency Distribution by Pitch Type (2023-06-01 to 2023-06-07)")
ax.set_ylim(0, 1.05)
ax.axhline(0.962, color="red", linestyle="--", alpha=0.5, label="league mean")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Full pipeline
agg, summary = build_pitcher_spin_profile(df)

print(f"pitcher × pitch_type rows: {len(agg)}")
print(f"pitcher summary rows:      {len(summary)}")
print()
summary.nlargest(10, "spin_efficiency_pct")[
    ["pitcher", "spin_efficiency_weighted", "spin_efficiency_pct",
     "dominant_pitch_type", "total_pitches"]
]

In [ ]:
# Reliability: one week is a small sample — most pitchers well below 200-pitch threshold
print("Reliability distribution:")
print(agg["reliability"].describe().round(3))
print()
print("Pitchers at full reliability (pitch_count >= 200 for a type):",
      (agg["reliability"] >= 1.0).sum())